In [14]:
# Importando las librerías
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os, json, time, tempfile
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.edge.options import Options
import time
import random
import pickle

In [23]:
def iniciar_driver():
    """Inicializa un driver de Edge con configuraciones para reducir la detección de automatización"""
    options = Options()
    # User-agent realista
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    # Desactiva banderas de automatización
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    # Inicia maximizado
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    driver = webdriver.Edge(options=options)
    driver.implicitly_wait(10)
    # Elimina la propiedad webdriver
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver

def pausa_humana(min_s=2, max_s=6):
    """Genera una pausa de duración aleatoria para simular comportamiento humano"""
    time.sleep(random.uniform(min_s, max_s))

In [25]:
def guardar_cookies(driver, ruta_archivo="cookies_foursquare.pkl"):
    """Guarda las cookies del navegador después de iniciar sesión manualmente"""
    pickle.dump(driver.get_cookies(), open(ruta_archivo, "wb"))
    print(f"Cookies guardadas en {ruta_archivo}")

def cargar_cookies(driver, ruta_archivo="cookies_foursquare.pkl"):
    """Carga cookies previamente guardadas"""
    try:
        cookies = pickle.load(open(ruta_archivo, "rb"))
        for cookie in cookies:
            # Algunos atributos pueden causar errores, los eliminamos
            if 'expiry' in cookie:
                del cookie['expiry']
            driver.add_cookie(cookie)
        return True
    except Exception as e:
        print(f"Error al cargar cookies: {e}")
        return False

In [26]:
def crear_sesion_inicial():
    """Inicia un navegador para que hagas login manualmente y guarda las cookies"""
    driver = iniciar_driver()
    driver.get("https://es.foursquare.com/login")
    print("1. Por favor, inicia sesión manualmente en Foursquare")
    print("2. Una vez iniciada la sesión correctamente, presiona Enter")
    input("Presiona Enter cuando hayas iniciado sesión...")
    guardar_cookies(driver)
    driver.quit()
    print("Sesión guardada. Ya puedes usar las cookies en tus scraping.")

# Descomenta la siguiente línea para crear o actualizar cookies
# crear_sesion_inicial()

In [27]:
def extraer_html_completo_con_cookies(url):
    """Extrae el HTML completo de una página usando autenticación por cookies"""
    driver = iniciar_driver()
    
    # Primero ir a la página base de Foursquare
    driver.get("https://es.foursquare.com/login")
    pausa_humana(1, 2)
    
    # Cargar cookies (autenticación guardada)
    if not cargar_cookies(driver):
        print("No se pudieron cargar las cookies. Ejecuta crear_sesion_inicial() primero.")
        driver.quit()
        return None
    
    driver.refresh()  # Refrescar para aplicar las cookies
    pausa_humana(2, 3)
    
    # Navegar a la URL deseada
    driver.get(url)
    pausa_humana(2, 4)
    
    # Comprobar si estamos autenticados correctamente
    if "login" in driver.current_url.lower() or "iniciar sesión" in driver.page_source.lower():
        print("La sesión no es válida o ha expirado. Ejecuta crear_sesion_inicial() de nuevo.")
        driver.quit()
        return None
    
    # Continuar con el scraping como antes
    while True:
        try:
            boton = driver.find_element(By.XPATH, '//button[contains(text(), "Ver más resultados")]')
            boton.click()
            pausa_humana(2, 4)
        except:
            break
    
    html = driver.page_source
    driver.quit()
    return html

In [28]:
def obtener_sitios_turisticos(html, url, json_file):
    # Agregar headers para simular un navegador real
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    
    # Realizar la solicitud HTTP
    response = requests.get(url, headers=headers)
    
    # Verificar si la solicitud fue exitosa
    if response.status_code == 200:
        # Parsear el contenido HTML
        soup = BeautifulSoup(html, 'html.parser')
        
        # Lista para almacenar los sitios turísticos como diccionarios
        sitios_list = []
        
        # Buscar todos los elementos que contienen información sobre sitios turísticos
        sitios = soup.find_all('div', class_='contentHolder')
        
        # Extraer información de cada sitio
        for i, sitio in enumerate(sitios):
            puntuacion_tag = sitio.find('div', class_='venueScore positive')
            nombre_tag = sitio.find('h2')
            categoria_tag = sitio.find('span', class_='venueDataItem')
            direccion_tag = sitio.find('div', class_='venueAddress')
            
            # --- Extracción de reseña, usuario, fecha y contenido ---
            reseña_contenedor = sitio.find('p', class_='tipText')
            usuario_reseña = "N/A"
            fecha_reseña = "N/A"
            contenido_reseña = "N/A"

            if reseña_contenedor:
                author_span = reseña_contenedor.find('span', class_='tipAuthor')
                if author_span:
                    usuario_reseña_tag = author_span.find('a', class_='userName')
                    if usuario_reseña_tag:
                        usuario_reseña = usuario_reseña_tag.get_text(strip=True)
                    # Extraer fecha (texto después del usuario y '•')
                    full_author_text = author_span.get_text(separator=' ', strip=True)
                    user_text = usuario_reseña_tag.get_text(strip=True) if usuario_reseña_tag else ""
                    potential_date_text = full_author_text.replace(user_text, '', 1).strip()
                    if potential_date_text.startswith('•'):
                        fecha_reseña = potential_date_text[1:].strip()
                    else:
                        fecha_reseña = potential_date_text
                # Extraer contenido de la reseña (texto fuera del span)
                full_tip_text = reseña_contenedor.get_text(separator=' ', strip=True)
                author_span_text = author_span.get_text(separator=' ', strip=True) if author_span else ""
                contenido_reseña = full_tip_text.replace(author_span_text, '', 1).strip()

            puntuacion = puntuacion_tag.get_text(strip=True) if puntuacion_tag else "N/A"
            nombre_link = nombre_tag.find('a') if nombre_tag else None
            nombre = nombre_link.get_text(strip=True) if nombre_link else (nombre_tag.get_text(strip=True) if nombre_tag else "N/A")
            categoria = categoria_tag.get_text(strip=True).replace('•', '').strip() if categoria_tag else "N/A"
            direccion = direccion_tag.get_text(strip=True) if direccion_tag else "N/A"
            url_sitio_tag = nombre_link if nombre_link else sitio.find('a')
            url_sitio = url_sitio_tag['href'] if url_sitio_tag and url_sitio_tag.has_attr('href') else ""
            if url_sitio.startswith('/'):
                url_sitio = requests.compat.urljoin(url, url_sitio)

            sitio_data = {
                "id": i + 1,
                "puntuacion": puntuacion,
                "nombre": nombre,
                "categoria": categoria,
                "direccion": direccion,
                "url_sitio": url_sitio,
                "usuario_reseña": usuario_reseña,
                "fecha_reseña": fecha_reseña,
                "contenido_reseña": contenido_reseña,
                "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S")
            }
            sitios_list.append(sitio_data)
        
        # Crear un diccionario con todos los datos
        datos = {
            "sitios_turisticos": sitios_list,
            "total": len(sitios_list),
            "fuente": url,
            "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S")
        }
        
        json_file_path = os.path.join(os.getcwd(),json_file)
        
        with open(json_file_path, 'w', encoding='utf-8') as f:
            json.dump(datos, f, ensure_ascii=False, indent=4)
        
        print(f"Datos guardados en archivo: {json_file_path}")
        print(f"Se encontraron {len(sitios_list)} sitios turísticos")
        
        return datos
    else:
        print(f"Error al acceder a la página: {response.status_code}")
        return None

In [29]:
url_Cartagena = "https://es.foursquare.com/explore?mode=url&near=Cartagena%20de%20Indias%2C%20Bol%C3%ADvar%2C%20Colombia&nearGeoId=72057594041615174"
url_SantaMarta = "https://es.foursquare.com/explore?mode=url&near=Santa%20Marta%2C%20Magdalena&nearGeoId=72057594041596541"
url_covenas = "https://es.foursquare.com/explore?mode=url&near=Cove%C3%B1as%2C%20Sucre%2C%20Colombia&nearGeoId=72057594041613648"

urls_archivos = [
    #(url_Cartagena, "sitios_turisticos_cartagena.json"),
    #(url_SantaMarta, "sitios_turisticos_santamarta.json"),
    (url_covenas, "sitios_turisticos_covenas.json")
]

In [21]:
# IMPORTANTE: Primero necesitas ejecutar esta celda para crear las cookies
# Descomenta la línea siguiente la primera vez o cuando necesites actualizar cookies
crear_sesion_inicial()

1. Por favor, inicia sesión manualmente en Foursquare
2. Una vez iniciada la sesión correctamente, presiona Enter
Cookies guardadas en cookies_foursquare.pkl
Sesión guardada. Ya puedes usar las cookies en tus scraping.


In [30]:
# Una vez que tengas las cookies guardadas, puedes ejecutar este bloque
for url, archivo in urls_archivos:
    html = extraer_html_completo_con_cookies(url)
    if html:
        obtener_sitios_turisticos(html, url, archivo)
    else:
        print(f"No se pudo extraer HTML de {url}")

Datos guardados en archivo: c:\Users\luisarias\Documents\webscrapping-foursquare\sitios_turisticos_covenas.json
Se encontraron 12 sitios turísticos


In [ ]:
def extraer_html_completo(url):
    driver = webdriver.Edge()  # O webdriver.Chrome(), etc.
    driver.get(url)
    time.sleep(3)
    while True:
        try:
            boton = driver.find_element(By.XPATH, '//button[contains(text(), "Ver más resultados")]')
            boton.click()
            time.sleep(2)
        except:
            break
    html = driver.page_source
    driver.quit()
    return html

In [3]:
def obtener_sitios_turisticos(html, url, json_file):
    # Agregar headers para simular un navegador real
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    
    # Realizar la solicitud HTTP
    response = requests.get(url, headers=headers)
    
    # Verificar si la solicitud fue exitosa
    if response.status_code == 200:
        # Parsear el contenido HTML
        soup = BeautifulSoup(html, 'html.parser')
        
        # Lista para almacenar los sitios turísticos como diccionarios
        sitios_list = []
        
        # Buscar todos los elementos que contienen información sobre sitios turísticos
        sitios = soup.find_all('div', class_='contentHolder')
        
        # Extraer información de cada sitio
        for i, sitio in enumerate(sitios):
            puntuacion_tag = sitio.find('div', class_='venueScore positive')
            nombre_tag = sitio.find('h2')
            categoria_tag = sitio.find('span', class_='venueDataItem')
            direccion_tag = sitio.find('div', class_='venueAddress')
            
            # --- Extracción de reseña, usuario, fecha y contenido ---
            reseña_contenedor = sitio.find('p', class_='tipText')
            usuario_reseña = "N/A"
            fecha_reseña = "N/A"
            contenido_reseña = "N/A"

            if reseña_contenedor:
                author_span = reseña_contenedor.find('span', class_='tipAuthor')
                if author_span:
                    usuario_reseña_tag = author_span.find('a', class_='userName')
                    if usuario_reseña_tag:
                        usuario_reseña = usuario_reseña_tag.get_text(strip=True)
                    # Extraer fecha (texto después del usuario y '•')
                    full_author_text = author_span.get_text(separator=' ', strip=True)
                    user_text = usuario_reseña_tag.get_text(strip=True) if usuario_reseña_tag else ""
                    potential_date_text = full_author_text.replace(user_text, '', 1).strip()
                    if potential_date_text.startswith('•'):
                        fecha_reseña = potential_date_text[1:].strip()
                    else:
                        fecha_reseña = potential_date_text
                # Extraer contenido de la reseña (texto fuera del span)
                full_tip_text = reseña_contenedor.get_text(separator=' ', strip=True)
                author_span_text = author_span.get_text(separator=' ', strip=True) if author_span else ""
                contenido_reseña = full_tip_text.replace(author_span_text, '', 1).strip()

            puntuacion = puntuacion_tag.get_text(strip=True) if puntuacion_tag else "N/A"
            nombre_link = nombre_tag.find('a') if nombre_tag else None
            nombre = nombre_link.get_text(strip=True) if nombre_link else (nombre_tag.get_text(strip=True) if nombre_tag else "N/A")
            categoria = categoria_tag.get_text(strip=True).replace('•', '').strip() if categoria_tag else "N/A"
            direccion = direccion_tag.get_text(strip=True) if direccion_tag else "N/A"
            url_sitio_tag = nombre_link if nombre_link else sitio.find('a')
            url_sitio = url_sitio_tag['href'] if url_sitio_tag and url_sitio_tag.has_attr('href') else ""
            if url_sitio.startswith('/'):
                url_sitio = requests.compat.urljoin(url, url_sitio)

            sitio_data = {
                "id": i + 1,
                "puntuacion": puntuacion,
                "nombre": nombre,
                "categoria": categoria,
                "direccion": direccion,
                "url_sitio": url_sitio,
                "usuario_reseña": usuario_reseña,
                "fecha_reseña": fecha_reseña,
                "contenido_reseña": contenido_reseña,
                "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S")
            }
            sitios_list.append(sitio_data)
        
        # Crear un diccionario con todos los datos
        datos = {
            "sitios_turisticos": sitios_list,
            "total": len(sitios_list),
            "fuente": url,
            "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S")
        }
        
        #project_dir = "c:/Users/luisarias/Documents/webscrapping-foursquare"
        json_file_path = os.path.join(os.getcwd(),json_file)
        
        with open(json_file_path, 'w', encoding='utf-8') as f:
            json.dump(datos, f, ensure_ascii=False, indent=4)
        
        print(f"Datos guardados en archivo: {json_file_path}")
        print(f"Se encontraron {len(sitios_list)} sitios turísticos")
        
        return datos
    else:
        print(f"Error al acceder a la página: {response.status_code}")
        return None

In [4]:
urls_archivos = [
    ("https://es.foursquare.com/explore?mode=url&near=Cartagena%20de%20Indias%2C%20Bol%C3%ADvar%2C%20Colombia&nearGeoId=72057594041615174", "sitios_turisticos_cartagena.json"),
    ("https://es.foursquare.com/explore?mode=url&near=Santa%20Marta%2C%20Magdalena&nearGeoId=72057594041596541", "sitios_turisticos_santamarta.json")
]

for url, archivo in urls_archivos:
    html = extraer_html_completo(url)
    obtener_sitios_turisticos(html, url, archivo)


Datos guardados en archivo: c:\Users\luisarias\Documents\webscrapping-foursquare\sitios_turisticos_cartagena.json
Se encontraron 0 sitios turísticos
Datos guardados en archivo: c:\Users\luisarias\Documents\webscrapping-foursquare\sitios_turisticos_santamarta.json
Se encontraron 0 sitios turísticos


In [ ]:
url_Cartagena= "https://es.foursquare.com/explore?mode=url&near=Cartagena%20de%20Indias%2C%20Bol%C3%ADvar%2C%20Colombia&nearGeoId=72057594041615174"